In [16]:
import pandas as pd
print("Pandas version:", pd.__version__)

Pandas version: 3.0.3


In [17]:
import os

print("current folder:", os.getcwd())
print("files in raw:")
for f in os.listdir("raw/"):
    print(" ", f)

current folder: c:\Users\hrikc\projects\arrears-analytics
files in raw:
  extract_20251222_01.csv
  extract_20251222_02.csv
  extract_20251222_03.csv
  real_sample_aemo_NEM12.csv


In [18]:
with open("raw/extract_20251222_01.csv") as f:
    lines = [line.strip() for line in f]

print(f"total lines: {len(lines)}")

from collections import Counter
record_types = [line.split(",")[0] for line in lines]
print(Counter(record_types))

total lines: 159
Counter({'300': 152, '200': 5, '100': 1, '900': 1})


In [19]:
df = pd.read_csv("raw/extract_20251222_01.csv", header=None)
df.head() 

ParserError: Error tokenizing data. C error: Expected 5 fields in line 2, saw 10


In [22]:
import pandas as pd

def parse_nem12(filepath):
    rows =[]
    # state variables — the "current 200 context"
    current_nmi = None
    current_suffix = None
    current_meter = None
    current_interval = None


    with open(filepath) as f:
        for line in f:
           fields = line.strip().split(",")
           record_type = fields[0]

           if record_type == "200":
              current_nmi = fields[1]
              current_suffix = fields[4]
              current_meter = fields[6]
              current_interval = fields[8]

           elif record_type == "300":
                date = fields[1]
                # the interval values sit between the date and the quality flag
                # TODO: how many values? where do they end?
                # append one row per reading, OR one row per day for now — your call
                n_values = 48 if current_interval == "30" else 96
                values = fields[2:2+n_values]
                # TODO: Create DataFrame rows for each value
                for i, value in enumerate(values):
                    rows.append({
                        "nmi": current_nmi,
                        "suffix": current_suffix,
                        "meter": current_meter,
                        "date": date,
                        "interval": i+1,
                        "reading": value
                    })

    return pd.DataFrame(rows)


In [20]:
df = parse_nem12("raw/extract_20251222_01.csv")
print(df.shape)
df.head(10)

(7296, 6)


,nmi,suffix,meter,date,interval,reading
0,6305000001,E1,MET00001,20251101,1,0.04
1,6305000001,E1,MET00001,20251101,2,0.04
2,6305000001,E1,MET00001,20251101,3,0.04
3,6305000001,E1,MET00001,20251101,4,0.04
4,6305000001,E1,MET00001,20251101,5,0.04
5,6305000001,E1,MET00001,20251101,6,0.04
6,6305000001,E1,MET00001,20251101,7,0.041
7,6305000001,E1,MET00001,20251101,8,0.042
8,6305000001,E1,MET00001,20251101,9,0.046
9,6305000001,E1,MET00001,20251101,10,0.056


In [27]:
import glob

all_files = glob.glob("raw/*.csv")
print("files found:", all_files)

#parse each, collect into DataFrames
frames = [parse_nem12(f) for f in all_files]

#Stack them all together into one big DataFrame
df = pd.concat(frames, ignore_index=True)
print("combined shape:", df.shape)
df.head()

files found: ['raw\\extract_20251222_01.csv', 'raw\\extract_20251222_02.csv', 'raw\\extract_20251222_03.csv', 'raw\\real_sample_aemo_NEM12.csv']
combined shape: (20592, 6)


,nmi,suffix,meter,date,interval,reading
0,6305000001,E1,MET00001,20251101,1,0.04
1,6305000001,E1,MET00001,20251101,2,0.04
2,6305000001,E1,MET00001,20251101,3,0.04
3,6305000001,E1,MET00001,20251101,4,0.04
4,6305000001,E1,MET00001,20251101,5,0.04


In [36]:
df["nmi_length"] = df["nmi"].str.len()
print(df["nmi_length"].value_counts())
print("Total number of nmis to be fixed", (df["nmi_length"] != 10).sum())

nmi_length
10    19104
9      1488
Name: count, dtype: int64
Total number of nmis to be fixed 1488


In [ ]:
df["reading_num"] = pd.to_numeric(df["reading"], errors="coerce")
print("negative readings:", (df["reading_num"] < 0).sum())

negative readings: 4


In [ ]:
print("non-numeric readings:unparseable readings", df["reading_num"].isna().sum())

non-numeric readings:unparseable readings 34


In [39]:
dupes = df.duplicated(subset=["nmi", "suffix", "meter", "date", "interval"], keep=False).sum()
print("duplicate rows:", dupes)

duplicate rows: 672
